# 32. Photometric Counterfactual 추론

같은 mask와 shape를 유지한 채 RGB appearance만 바꾸어 `red` 실패가 입력 색상 shortcut 때문인지 확인합니다.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "ch3_utils.py").exists():
    matches = list(Path.cwd().glob("Deeplearning/*/3장/ch3_utils.py")) + list(Path.cwd().glob("**/ch3_utils.py"))
    if matches:
        NOTEBOOK_DIR = matches[0].parent
    else:
        NOTEBOOK_DIR = Path("Deeplearning") / "Vision 응용" / "3장"
sys.path.insert(0, str(NOTEBOOK_DIR))

from ch3_utils import *

paths = find_ch3_paths()
set_korean_font()
set_seed(31)
paths

Chapter3Paths(chapter3_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장'), chapter2_2_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-2장'), data_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/data'), stress_ladder_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/data/synthetic_metal_stress_ladder'), runs_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/runs'), manifest_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/runs/manifests'), ch2_2_runs_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-2장/runs'))

## 32-1. 모델과 manifest 준비

In [2]:
samples = load_ch3_base_samples()
manifests = create_ch3_probe_manifests(samples, max_per_cell=None, seed=31)
registry = discover_ch3_model_registry()

MODEL_VARIANT = "baseline_no_aug"
MODEL_SEED = 0
MAX_SAMPLES = None
transforms = DEFAULT_COUNTERFACTUALS

selected = registry[
    (registry["variant"] == MODEL_VARIANT)
    & (registry["model_seed"] == MODEL_SEED)
    & (registry["checkpoint_exists"])
]
if selected.empty:
    raise FileNotFoundError("선택한 모델 checkpoint가 없습니다.")
model_run_dir = Path(selected.iloc[0]["run_dir"])
model_run_dir

WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-2장/runs/baseline_seed_repeats/seed_0')

## 32-2. Counterfactual 추론 실행

In [3]:
out_dir = paths.runs_root / "photometric_counterfactual"
metrics = evaluate_counterfactual_manifest(
    model_run_dir,
    manifests["eval_color_counterfactual_probe"],
    out_dir,
    transforms=transforms,
    max_samples=MAX_SAMPLES,
    seed=31,
)
plot_counterfactual_sensitivity(metrics, out_dir / "photometric_sensitivity_curves.png")
display(metrics.head())
display(
    metrics.groupby(["transform", "color_group"])[["target_dice", "target_fnr", "prediction_flip_rate"]]
    .mean()
    .reset_index()
)

C:\Users\준승\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,sample_id,model_run_dir,transform,color_group,shape_group,defect_type,target_dice,target_iou,target_fnr,target_fpr,target_margin,entropy_mean,prediction_flip_rate
0,eval_matched_matched_control_neutral_scratch_t...,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,original,neutral,top_half_metal,scratch,0.714286,0.555556,0.257812,0.002645,0.834934,0.027422,0.000000
1,eval_matched_matched_control_neutral_scratch_t...,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,grayscale,neutral,top_half_metal,scratch,0.708487,0.548571,0.250000,0.002891,0.999306,0.027930,0.001099
2,eval_matched_matched_control_neutral_scratch_t...,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,gray_world,neutral,top_half_metal,scratch,0.708955,0.549133,0.257812,0.002768,0.848294,0.027183,0.000977
3,eval_matched_matched_control_neutral_scratch_t...,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,red_to_neutral,neutral,top_half_metal,scratch,0.683824,0.519553,0.273438,0.003137,0.767610,0.028697,0.003235
4,eval_matched_matched_control_neutral_scratch_t...,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,hue_rotate_90,neutral,top_half_metal,scratch,0.694030,0.531429,0.273438,0.002891,0.800798,0.024747,0.003479


,transform,color_group,target_dice,target_fnr,prediction_flip_rate
0,brightness_0p75,blue,0.646608,0.374652,0.004091
1,brightness_0p75,neutral,0.643196,0.396532,0.004220
2,brightness_0p75,purple,0.409137,0.637001,0.009383
3,brightness_0p75,red,0.000000,1.000000,0.172890
4,brightness_1p25,blue,0.590802,0.435185,0.006068
5,brightness_1p25,neutral,0.581520,0.450116,0.021884
6,brightness_1p25,purple,0.247844,0.762189,0.010199
7,brightness_1p25,red,0.012953,0.992475,0.033056
8,channel_shuffle_bgr,blue,0.395861,0.659475,0.014336
9,channel_shuffle_bgr,neutral,0.623224,0.394160,0.003091


## 32-3. photometric augmentation 모델과 비교

In [4]:
RUN_PHOTOMETRIC_COMPARISON = True

if RUN_PHOTOMETRIC_COMPARISON:
    photo = registry[
        (registry["variant"] == "photometric_aug")
        & (registry["model_seed"] == MODEL_SEED)
        & (registry["checkpoint_exists"])
    ]
    if photo.empty:
        raise FileNotFoundError("photometric_aug 모델이 없습니다. 2-2장 27번 노트북을 먼저 실행하세요.")
    photo_out = paths.runs_root / "photometric_counterfactual" / "photometric_aug_seed_0"
    photo_metrics = evaluate_counterfactual_manifest(
        photo.iloc[0]["run_dir"],
        manifests["eval_color_counterfactual_probe"],
        photo_out,
        transforms=transforms,
        max_samples=MAX_SAMPLES,
        seed=31,
    )
    display(
        photo_metrics.groupby(["transform", "color_group"])[["target_dice", "target_fnr"]]
        .mean()
        .reset_index()
    )

,transform,color_group,target_dice,target_fnr
0,brightness_0p75,blue,0.616964,0.411609
1,brightness_0p75,neutral,0.631359,0.418035
2,brightness_0p75,purple,0.657311,0.362621
3,brightness_0p75,red,0.273805,0.760078
4,brightness_1p25,blue,0.605223,0.425710
5,brightness_1p25,neutral,0.648748,0.399535
6,brightness_1p25,purple,0.579781,0.462725
7,brightness_1p25,red,0.370212,0.663272
8,channel_shuffle_bgr,blue,0.618717,0.402026
9,channel_shuffle_bgr,neutral,0.654644,0.389529
